<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 3 (Miércoles) — Clasificación 2 — Redes neuronales artificiales**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🧠 Red neuronal artificial: teoría y construcción desde cero (enfermedad cardíaca)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Entender la neurona artificial, las capas, la función de activación, la pérdida y la retropropagación. |
| 🔍 Comprensión de los datos | Cargamos `heart.csv` (303 pacientes, 13 variables clínicas) y la variable objetivo `target`. |
| 🧹 Preparación de los datos | Dividimos 80/20 y estandarizamos usando solo el entrenamiento. |
| 🤖 Modelado | Programamos una red de dos capas (ReLU + sigmoide) con descenso de gradiente y retropropagación. |
| 📊 Evaluación | Evaluamos con exactitud, curva ROC (con probabilidades), matriz de confusión, sensibilidad y especificidad. |
| 🚀 Despliegue y conclusiones | Verificamos el gradiente numéricamente y comparamos con `scikit-learn`. |

> 🎯 **Analogía:** un equipo de evaluadores en cadena: cada neurona pondera las pistas que recibe (peso), añade su criterio base (sesgo) y pasa su opinión a la siguiente capa; cuando la red se equivoca, el error se reparte hacia atrás para ajustar los pesos.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 👉 **3** | Miércoles | **Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales** |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('heart.csv', 'dia_4')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Entender la neurona artificial, las capas, la función de activación, la pérdida y la retropropagación.

📚 Lectura recomendada: [3] [4] [6]

# **Redes neuronales artificiales**

Las redes neuronales artificiales (en inglés, ANN, Artificial Neural Networks) se inspiran en el sistema nervioso y el comportamiento biológico, creando un sistema de interconexión en capas de neuronas artificiales que colaboran para procesar datos de entrada y generar salidas

![Perceptrón multicapa: capa de entrada, capa oculta y capa de salida](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_02.png)
*Figura: Perceptrón multicapa: capa de entrada, capa oculta y capa de salida. [14]*

**Historia**

El primer modelo matemático de neurona lo propusieron en 1943 el neurofisiólogo Warren McCulloch y el matemático Walter Pitts [3]. En 1958 Frank Rosenblatt presentó el **perceptrón** [4], pero en 1969 Minsky y Papert mostraron que una sola neurona no puede resolver problemas no lineales como el XOR [5], lo que enfrió el interés. El campo revivió en 1986 cuando se popularizó la **retropropagación** para entrenar redes de varias capas [6]. En los años 90 técnicas como las SVM las superaron en muchas tareas y, desde 2012, con más datos y tarjetas GPU, el aprendizaje profundo volvió a dominar la visión por computador [7].

![Neurona artificial: entradas, pesos, suma ponderada, activación y salida](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_01.png)
*Figura: Neurona artificial: entradas, pesos, suma ponderada, activación y salida. [14]*

Una neurona artificial se compone de una o múltiples entradas, un peso, un bias, un sumador y una función de transferencia (o de activación), como se muestra en la figura anterior. Normalmente una neurona, aún con múltiples entradas, no es suficiente, por lo tanto, las ANNs usan varias neuronas ubicadas en paralelo formando lo que se conoce como una capa




En el caso de las neuronas artificiales, la suma de las entradas multiplicadas por sus pesos asociados determina el “impulso nervioso” que recibe la neurona. Este valor, se procesa en el interior de la célula mediante una función de activación que devuelve un valor que se envía como salida de la neurona.

Del mismo modo que nuestro cerebro está compuesto por neuronas interconectadas entre sí, una red neuronal artificial está formada por neuronas artificiales conectadas entre sí y agrupadas en diferentes niveles que denominamos capas:

» Una capa es un conjunto de neuronas cuyas entradas provienen de una capa anterior (o de los datos de entrada en el caso de la primera capa) y cuyas salidas son la entrada de una capa posterior.»

$$$$

**En la siguiente imagen podemos ver una red con cuatro capas:**

![Red de tres capas (L = 3) con sesgos y pesos entre capas](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_03.png)
*Figura: Red de tres capas (L = 3) con sesgos y pesos entre capas. [14]*

$$$$

**Visualización de los pesos entre las capas**



Las neuronas de la primera capa reciben como entrada los datos reales que alimentan a la red neuronal.  Es por eso por lo que la primera capa se conoce como capa de entrada. La salida de la última capa es el resultado visible de la red, por lo que la última capa se conoce como la capa de salida. Las capas que se sitúan entre la capa de entrada y la capa de salida se conocen como capas ocultas ya que desconocemos tanto los valores de entrada como los de salida.

$$$$

**Arquitectura de una red neuronal artificial**




La primera capa o capa de entrada (input layer) tiene nodos de entrada que envían datos a una segunda capa. Estos nodos son pasivos y simplemente transmiten la información a la siguiente capa. El número de nodos de esta capa coincide con la cantidad de datos que se introducen.

Los nodos de la segunda capa o capa oculta (hidden layer) filtran los patrones relevantes de los que no lo son identificando la información importante. Estos nodos son activos, lo que significa que combinan los datos procedentes de la capa anterior. Cada input recibido es multiplicado por un peso y los resultantes son sumados entre sí y delimitados con una función (sigmoide o logística) para mejorar la eficiencia. Estos nodos suelen representar un 10% con respecto a los de la primera capa.

En la tercera capa, también llamada capa de salida (output layer) se repite el proceso de la segunda y los datos se vuelven a combinar y modificar en los nodos activos para producir los valores de salida.




**La capa de entrada**: representa las variables de entrada más el término de sesgo. Por lo tanto, si hay n variables de entrada, el tamaño de la capa de entrada es n + 1, donde + 1 es el término de sesgo

**La capa / capas ocultas**: significan neuronas donde se realizan todos los cálculos matemáticos. Tenga en cuenta que una red neuronal determinada puede tener más de una neurona en una capa oculta o también varias capas ocultas

**La función de activación**: convierte la salida de una capa determinada antes de pasar la información a capas consecutivas. Las funciones de activación son ecuaciones matemáticas que determinan la salida de una red neuronal dada. Es una parte de cada neurona en las capas ocultas y determina la salida relevante para la predicción.

**La capa de salida**: la "predicción de salida" final de la red.
Propagación hacia adelante: cálculo de la salida de cada iteración desde la capa de entrada a la capa de salida

**Propagación hacia atrás**: calcula las ponderaciones revisadas (w1, w2, w3 y b1) después de cada propagación hacia adelante mediante el análisis de la derivada de la función de costo utilizada para optimizar la salida del modelo.

**Tasa de aprendizaje**: determinó el cambio porcentual atribuido a cada peso y término de sesgo después de cada propagación hacia atrás, es decir, controla la velocidad a la que el modelo aprende información sobre los datos.


**Entonces, ¿qué sucede en una red neuronal?**
La idea de un algoritmo ANN es determinar los valores óptimos de pesos y términos de sesgo de manera que la función de error sea lo más mínima posible.

$$$$

Por eso

**Paso 1**: los pesos w1, w2, w3 y b1 se inicializan con algunos valores aleatorios

**Paso 2**: las capas ocultas utilizan los pesos y las variables de entrada dados para calcular la puntuación de probabilidad. Las funciones de activación (por ejemplo, sigmoidea) se aplican si corresponde

**Paso 3**: la función de costo se pone en uso y el algoritmo de descenso de gradiente se repite para calcular los pesos actualizados. Los pesos actualizados se determinan como peso (w) = w-alfa * (dJ / dw), donde alfa es la tasa de aprendizaje y dJ / dw es el cambio en la función de costo cuando cambia el peso. De manera similar, sesgo (b) = b-alpha * (dJ / db)

**Paso 4**: los pasos 2 y 3 se repiten hasta que la función de costo se minimiza

$$$$

**Perceptron**



**Feed Forward**

![Propagación hacia adelante: la información fluye de la entrada a la salida](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_11.png)
*Figura: Propagación hacia adelante: la información fluye de la entrada a la salida. [14]*

$$$$

**Backpropagation**

![Retropropagación: el error de salida se reparte hacia las capas anteriores](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_12.png)
*Figura: Retropropagación: el error de salida se reparte hacia las capas anteriores. [14]*


### **Las capas de una red neuronal**

La capa es una colección de nodos en diferentes etapas de cálculo en una red neuronal. Cada nodo actúa como una neurona y realiza cálculos sobre los datos que se le pasan.



**Los pesos y sesgos**

Son los parámetros que se pueden aprender y que ayudan a una red neuronal a aprender correctamente una función.
Piense en los pesos como una medida de qué tan seguro está de que una característica contribuye a una predicción y el sesgo como un valor base desde el que deben comenzar sus predicciones.

Ejemplo
Suponga que es un modelo de aprendizaje automático y desea predecir si una persona es rica o no, y se le han dado las siguientes pistas para ayudarlo a tomar esa decisión:



*   Edad de la persona
*   Altura de la persona
*   Salario de la persona

Las pistas anteriores son lo que llamamos características en el aprendizaje automático, y lo que desea predecir se llama verdad objetivo / etiqueta / terreno.

La etiqueta puede ser una de dos clases (rica, no rica), en otras palabras, clasificación binaria.

Básicamente, lo que desea hacer es combinar las funciones de tal manera que lo ayuden a predecir con mayor precisión el resultado.

$$$$
y(rico, no rico) = Edad + Altura + salario + base
$$$$

Suponiendo que establezcamos un salario base de 3000 y la Persona 1 tenga las siguientes características; edad = 18, altura = 5,6 pies, salario = 2000, luego calculará la riqueza de la siguiente manera:
y (rico, no rico) = 18 + 5.6 + 2000 + 3000 = ~ 5024

$$$$
Para este ejemplo, podríamos definir un umbral de riqueza como cualquier valor superior a 40,000. A juzgar por estos criterios, puede concluir que la persona 1 no es rica.

$$$$
Veamos otro ejemplo.

La Persona 2 tiene las siguientes características; edad = 26, altura = 5.2 pies y salario = 50,000. Su predicción se calculará como:
y (rico, no rico) = 26 + 5.2 + 50,000 + 3000 = ~ 53031

Entonces, según el umbral indicado anteriormente, la persona 2 es rica.
$$$$

Es obvio que algunas pistas son más importantes que otras. ¿Puedes adivinar cuál es el más importante? ¡Sí! Salario.

Este es, quizás como era de esperar, un factor importante que indica si una persona es rica o no.

Con esta idea, puede asignar importancia a las características. Por ejemplo, puede asignar pesos de la siguiente manera:
y (rico, no rico) = (2 * Edad) + (1 * Altura) + (8 * Salario) + base


De manera intuitiva, asignamos un valor más alto a la función de salario.
La importancia del valor puede ser cualquier número pero debe ser representativo de la escala.


Quizás se pregunte cuál es el valor base de 3000 y por qué lo agregamos a las predicciones. Este valor se llama sesgo. Es un valor base que toda predicción debe tener, incluso cuando no se da nada más.

Ahora, si vuelve a hacer una predicción para la persona 1 y 2, tendrá lo siguiente:


**Persona 1: (2 * 18) + (1 * 5.6) (8 * 2000) + 3000 = ~ 19041 (todavía pobre)**

$$$$
**Persona 2: (2 * 26) + (1 * 5.2) + (8 * 5000) + 3000 = ~ 43057 (Todavía rico)**
$$$$
¿Qué pasa si una persona no tiene valor por la edad, la altura y el salario, entonces su predicción será?

**y (rico, no rico) = (2 * 0) + (1 * 0) (8 * 0) + 3000 = 3000**

Ahora ve dónde entra el valor de sesgo.


**La función de activación**

Primero, hace una suma ponderada de la entrada y los pesos, agrega los sesgos y luego pasa el resultado a través de una función de activación.

Una función de activación es lo que hace que una red neuronal sea capaz de aprender funciones complejas no lineales. Las funciones no lineales son difíciles de aprender para los algoritmos tradicionales de aprendizaje automático, como la regresión logística y lineal. La función de activación es lo que hace que una red neuronal sea capaz de comprender estas funciones.
Hay muchos tipos de funciones de activación que se utilizan en el aprendizaje profundo; algunas populares son Sigmoid, ReLU, tanh, Leaky ReLU, etc.

![Función sigmoide: convierte cualquier valor z en una probabilidad entre 0 y 1](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_02.png)
*Figura: Función sigmoide: convierte cualquier valor z en una probabilidad entre 0 y 1. [14]*

La función de salida dependerá de lo que intente predecir. Puede usar una función **sigmoide** cuando tiene un problema de dos clases (clasificación binaria) y puede usar una función llamada **softmax** para problemas de varias clases..



**La función de pérdida**

ES una forma de medir qué tan buena es la predicción de un modelo para que pueda ajustar las ponderaciones y sesgos.
Una función de pérdida debe diseñarse adecuadamente para que pueda penalizar correctamente a un modelo incorrecto y recompensar a un modelo correcto. Esto significa que desea que la pérdida le diga si una predicción realizada está lejos o cerca de la predicción verdadera. La elección de la función de pérdida depende de la tarea y, para problemas de clasificación, puede utilizar la pérdida de entropía cruzada.

**Propagación hacia adelante / Feed Forward **

La propagación hacia adelante es el nombre que se le da a la serie de cálculos realizados por la red neuronal antes de realizar una predicción. En su red de dos capas, realizará el siguiente cálculo para la propagación hacia adelante:
$$$$



*   Calcule la suma ponderada entre la entrada y los pesos de la primera capa y luego agregue el sesgo: Z1 = (W1 * X) + b
*   Pasar el resultado a través de la función de activación ReLU: A1 = Relu (Z1)
*   Calcule la suma ponderada entre la salida (A1) del paso anterior y los pesos de la segunda capa; agregue también el sesgo: Z2 = (W2 * A1) + b2
*   Calcule la función de salida pasando el resultado a través de una función sigmoidea: A2 = sigmoidea (Z2)
*   Y finalmente, calcule la pérdida entre la salida prevista y las etiquetas verdaderas: pérdida (A2, Y)


**Retropropagación / Backpropagation**

Una red neuronal aprende a predecir los valores correctos probando continuamente diferentes valores para los pesos y luego comparando las pérdidas. Si la función de pérdida disminuye, entonces el peso actual es mejor que el anterior, o viceversa. Esto significa que la red neuronal tiene que pasar por muchos ciclos de entrenamiento (propagación hacia adelante) y actualización (propagación hacia atrás) para obtener los mejores pesos y sesgos. Este ciclo es lo que generalmente llamamos la fase de entrenamiento, y el proceso de búsqueda de los pesos correctos se llama optimización.

Después de calcular la salida y la pérdida en la capa de propagación hacia adelante, pasará a la fase de propagación hacia atrás, donde calcula las derivadas hacia atrás, desde la pérdida hasta el primer peso y sesgo. Para realizar la propagación hacia atrás en su red neuronal, deberá seguir los pasos que se enumeran a continuación:

A partir de la última capa, calcule la derivada de la pérdida con respecto a la salida yhat como:

$$
d l_{-} w r t_{-} Y h a t=\frac{-y}{\hat{y}}+\frac{1-y}{1-\hat{y}}
$$

**Learning rate**

La tasa de aprendizaje es un hiperparámetro que controla cuánto cambiar el modelo en respuesta al error estimado cada vez que se actualizan los pesos del modelo. Elegir la tasa de aprendizaje es un desafío, ya que un valor demasiado pequeño puede resultar en un proceso de entrenamiento largo que podría atascarse, mientras que un valor demasiado grande puede resultar en aprender un conjunto de pesos subóptimo demasiado rápido o un proceso de entrenamiento inestable.

![Función de costo que disminuye a lo largo de las épocas de entrenamiento](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_07.png)
*Figura: Función de costo que disminuye a lo largo de las épocas de entrenamiento. [14]*

![Superficie de costo con mínimos locales y un mínimo global](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch12/images/12_13.png)
*Figura: Superficie de costo con mínimos locales y un mínimo global. [14]*






























---
## 🤖 Fase 4 · Modelado — red neuronal programada desde cero

Programamos una red de dos capas (ReLU + sigmoide) con descenso de gradiente y retropropagación.

📚 Lectura recomendada: [6] [8]

**Nota sobre el código que sigue.** La clase `NeuralNet` suma (no promedia) los gradientes de las muestras, por lo que la tasa de aprendizaje efectiva es proporcional al tamaño de la muestra; por eso se usan valores pequeños (0.001–0.01). La pérdida que se grafica sí es un promedio [8].

## Modelado desde 0 ANN


In [ ]:
import csv
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore") #suppress warnings
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [ ]:
class NeuralNet():
    '''
    A two layer neural network
    '''

    def __init__(self, layers=[13,2,1], learning_rate=0.001, iterations=100):
        self.params = {}
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.loss = []
        self.sample_size = None
        self.layers = layers
        self.X = None
        self.y = None

    def init_weights(self):
        '''
        Initialize the weights from a random normal distribution
        '''
        np.random.seed(1) # Seed the random number generator
        self.params["W1"] = np.random.randn(self.layers[0], self.layers[1])
        self.params['b1']  =np.random.randn(self.layers[1],)
        self.params['W2'] = np.random.randn(self.layers[1],self.layers[2])
        self.params['b2'] = np.random.randn(self.layers[2],)

    def relu(self,Z):
        '''
        The ReLu activation function is to performs a threshold
        operation to each input element where values less
        than zero are set to zero.
        '''
        return np.maximum(0,Z)

    def dRelu(self, x):
        return (x > 0).astype(float)   # derivada de ReLU (sin modificar el arreglo original)

    def eta(self, x):
      ETA = 0.0000000001
      return np.maximum(x, ETA)


    def sigmoid(self,Z):
        '''
        The sigmoid function takes in real numbers in any range and
        squashes it to a real-valued output between 0 and 1.
        '''
        return 1/(1+np.exp(-Z))

    def entropy_loss(self,y, yhat):
        nsample = len(y)
        yhat_inv = 1.0 - yhat
        y_inv = 1.0 - y
        yhat = self.eta(yhat) ## clips value to avoid NaNs in log
        yhat_inv = self.eta(yhat_inv)
        loss = -1/nsample * (np.sum(np.multiply(np.log(yhat), y) + np.multiply((y_inv), np.log(yhat_inv))))
        return loss

    def forward_propagation(self):
        '''
        Performs the forward propagation
        '''

        Z1 = self.X.dot(self.params['W1']) + self.params['b1']
        A1 = self.relu(Z1)
        Z2 = A1.dot(self.params['W2']) + self.params['b2']
        yhat = self.sigmoid(Z2)
        loss = self.entropy_loss(self.y,yhat)

        # save calculated parameters
        self.params['Z1'] = Z1
        self.params['Z2'] = Z2
        self.params['A1'] = A1

        return yhat,loss

    def back_propagation(self,yhat):
        '''
        Computes the derivatives and update weights and bias according.
        '''
        y_inv = 1 - self.y
        yhat_inv = 1 - yhat

        dl_wrt_yhat = np.divide(y_inv, self.eta(yhat_inv)) - np.divide(self.y, self.eta(yhat))
        dl_wrt_sig = yhat * (yhat_inv)
        dl_wrt_z2 = dl_wrt_yhat * dl_wrt_sig

        dl_wrt_A1 = dl_wrt_z2.dot(self.params['W2'].T)
        dl_wrt_w2 = self.params['A1'].T.dot(dl_wrt_z2)
        dl_wrt_b2 = np.sum(dl_wrt_z2, axis=0, keepdims=True)

        dl_wrt_z1 = dl_wrt_A1 * self.dRelu(self.params['Z1'])
        dl_wrt_w1 = self.X.T.dot(dl_wrt_z1)
        dl_wrt_b1 = np.sum(dl_wrt_z1, axis=0, keepdims=True)

        #update the weights and bias
        self.params['W1'] = self.params['W1'] - self.learning_rate * dl_wrt_w1
        self.params['W2'] = self.params['W2'] - self.learning_rate * dl_wrt_w2
        self.params['b1'] = self.params['b1'] - self.learning_rate * dl_wrt_b1
        self.params['b2'] = self.params['b2'] - self.learning_rate * dl_wrt_b2

    def fit(self, X, y):
        '''
        Trains the neural network using the specified data and labels
        '''
        self.X = X
        self.y = y
        self.init_weights() #initialize weights and bias


        for i in range(self.iterations):
            yhat, loss = self.forward_propagation()
            self.back_propagation(yhat)
            self.loss.append(loss)

    def predict(self, X):
        '''
        Predicts on a test data
        '''
        Z1 = X.dot(self.params['W1']) + self.params['b1']
        A1 = self.relu(Z1)
        Z2 = A1.dot(self.params['W2']) + self.params['b2']
        pred = self.sigmoid(Z2)
        return np.round(pred)

    def predict_proba(self, X):
        '''Probabilidad de la clase 1 (salida de la sigmoide), necesaria para la curva ROC'''
        A1 = self.relu(X.dot(self.params['W1']) + self.params['b1'])
        return self.sigmoid(A1.dot(self.params['W2']) + self.params['b2'])

    def acc(self, y, yhat):
        '''
        Calculates the accutacy between the predicted valuea and the truth labels
        '''
        acc = float(np.mean(y == yhat) * 100)   # corregido: sum() devolvía un arreglo y int() fallaba
        return acc


    def plot_loss(self):
        '''
        Plots the loss curve
        '''
        plt.plot(self.loss)
        plt.xlabel("Iteration")
        plt.ylabel("logloss")
        plt.title("Loss curve for training")
        plt.show()

---
## 🔍 Fase 2 · Comprensión de los datos — cargar heart.csv

Cargamos `heart.csv` (303 pacientes, 13 variables clínicas) y la variable objetivo `target`.

📚 Lectura recomendada: [9] [10]

In [ ]:
# Leer la información
heart_df = pd.read_csv('sample_data/heart.csv', sep=',')

In [ ]:
heart_df

---
## 🧹 Fase 3 · Preparación de los datos — estandarizar y dividir

Dividimos 80/20 y estandarizamos usando solo el entrenamiento.

📚 Lectura recomendada: [15]

In [ ]:
y_label=heart_df['target'].values.reshape(heart_df.shape[0], 1)
X = heart_df.drop(columns=['target'])

#le = preprocessing.LabelEncoder()
#y_label =le.fit_transform(y_label)


#split data into train and test set
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y_label, test_size=0.2, random_state=2)

#standardize the dataset
sc = StandardScaler()
sc.fit(Xtrain)
Xtrain = sc.transform(Xtrain)
Xtest = sc.transform(Xtest)


print(f"Shape of train set is {Xtrain.shape}")
print(f"Shape of test set is {Xtest.shape}")
print(f"Shape of train label is {ytrain.shape}")
print(f"Shape of test labels is {ytest.shape}")

---
## 🤖 Fase 4 · Modelado — entrenar la red

Programamos una red de dos capas (ReLU + sigmoide) con descenso de gradiente y retropropagación.

In [ ]:
ann = NeuralNet() # create the NN model
ann.fit(Xtrain, ytrain) #train the model

In [ ]:
ann.plot_loss()

In [ ]:
ann = NeuralNet(layers=[13,2,1], learning_rate=0.01, iterations=500) # create the NN model
ann.fit(Xtrain, ytrain) #train the model
ann.plot_loss()


In [ ]:
test_pred = ann.predict(Xtest)

print("Exactitud en prueba: {:.1f} %".format(ann.acc(ytest, test_pred)))

---
## 📊 Fase 5 · Evaluación

Evaluamos con exactitud, curva ROC (con probabilidades), matriz de confusión, sensibilidad y especificidad.

📚 Lectura recomendada: [12] [13]

**Evaluaciones de los modelos de clasificación**

In [ ]:
predictions = pd.DataFrame(ann.predict(Xtest))
probs = ann.predict_proba(Xtest)      # la curva ROC y el AUC se calculan con probabilidades, no con etiquetas 0/1
auc = roc_auc_score(ytest, probs)
print('AUC: %.2f' % auc)

In [ ]:
fpr, tpr, thresholds = roc_curve(ytest, probs)
plt.plot(fpr, tpr, color="orange", label="ROC")
plt.plot([0, 1], [0, 1], color="darkblue", linestyle="--")
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curva ROC de la red neuronal')
plt.legend()
plt.show()

### **Matriz de confusión**

Es una herramienta que permite la visualización del desempeño de un algoritmo que se emplea en aprendizaje supervisado. Cada columna de la matriz representa el número de predicciones de cada clase, mientras que cada fila representa a las instancias en la clase real. Uno de los beneficios de las matrices de confusión es que facilitan ver si el sistema está confundiendo dos clases.


![Matriz de confusión: VP, FN, FP y VN](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_08.png)
*Figura: Matriz de confusión: VP, FN, FP y VN. [14]*

**Errores**

$$ $$
![Ejemplo de matriz de confusión con conteos reales](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_09.png)
*Figura: Ejemplo de matriz de confusión con conteos reales. [14]*

 **Notas Importantes**









**Exactitud**: se refiere a lo cerca que está el resultado de una medición del valor verdadero. En términos estadísticos, la exactitud está relacionada con el sesgo de una estimación. No debe confundirse con la sensibilidad (tasa de verdaderos positivos).  Se representa como  la proporción de resultados verdaderos (tanto verdaderos positivos (VP) como verdaderos negativos (VN)) dividido entre el número total de casos

$$ $$

\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}
$$ $$

**Sensibilidad**: se conoce como Tasa de Verdaderos Positivos (True Positive Rate) ó TP. Es la proporción de casos positivos que fueron correctamente identificadas por el algoritmo

$$$$
\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

$$$$

**Especificidad**: También conocida como la Tasa de Verdaderos Negativos, (“true negative rate”) o TN. Se trata de los casos negativos que el algoritmo ha clasificado correctamente.  


\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}
$$ $$

**Ejemplo**

In [ ]:

# crear matriz de confusión
matrix = confusion_matrix(ytest, predictions)

# crear mapa de calor dibujar mapa de calor
sns.heatmap(matrix, annot=True, cbar=None, cmap="Blues")
plt.title("Matriz de confusión"), plt.tight_layout()
plt.ylabel("Clases verdaderas"), plt.xlabel("Clase clasificada")
plt.show()

matrix

\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}

In [ ]:
VN, FP, FN, VP = matrix.ravel()
(VP + VN) / matrix.sum()

\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

In [ ]:
VP / (VP + FN)   # sensibilidad

\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}

In [ ]:
VN / (VN + FP)   # especificidad

---
## 🧪 Refuerzo: comprobar la retropropagación con el gradiente numérico y comparar con scikit-learn

Programar la retropropagación a mano es propenso a errores. La prueba estándar es el **chequeo numérico del gradiente**: la derivada analítica que calcula `back_propagation` debe coincidir con la pendiente aproximada $\frac{L(w+h)-L(w-h)}{2h}$ [8]. Después comparamos nuestra red con la de `scikit-learn` y vemos cómo cambia la exactitud con el número de neuronas ocultas.

In [ ]:
# 1) Chequeo numérico del gradiente de W2 (con learning_rate = 1 la variación de W2 es justo el gradiente de la suma de pérdidas)
red = NeuralNet(layers=[13, 4, 1], learning_rate=1.0)
red.X, red.y = Xtrain, ytrain
red.init_weights()
W0 = {k: v.copy() for k, v in red.params.items()}
yhat, _ = red.forward_propagation()
red.back_propagation(yhat)
grad_analitico = W0['W2'] - red.params['W2']

def perdida_total(W2_nuevo):
    r = NeuralNet(layers=[13, 4, 1]); r.X, r.y = Xtrain, ytrain; r.init_weights(); r.params['W2'] = W2_nuevo
    return r.forward_propagation()[1] * len(Xtrain)

h = 1e-5
for j in range(4):
    mas, menos = W0['W2'].copy(), W0['W2'].copy(); mas[j, 0] += h; menos[j, 0] -= h
    numerico = (perdida_total(mas) - perdida_total(menos)) / (2 * h)
    print(f'dL/dW2[{j}]:  analítico = {grad_analitico[j, 0]:10.5f}   numérico = {numerico:10.5f}')

# 2) Nuestra red vs. MLPClassifier de scikit-learn, variando las neuronas ocultas
from sklearn.neural_network import MLPClassifier
filas = []
for ocultas in (2, 4, 8, 16):
    propia = NeuralNet(layers=[13, ocultas, 1], learning_rate=0.001, iterations=500); propia.fit(Xtrain, ytrain)
    sk = MLPClassifier(hidden_layer_sizes=(ocultas,), max_iter=2000, random_state=1).fit(Xtrain, ytrain.ravel())
    filas.append({'neuronas ocultas': ocultas, 'exactitud red propia': round(ann.acc(ytest, propia.predict(Xtest)) / 100, 3), 'exactitud sklearn': round(sk.score(Xtest, ytest.ravel()), 3)})
pd.DataFrame(filas)

### ✍️ Ejercicios
1. ¿Coinciden el gradiente analítico y el numérico? ¿Qué indicaría una diferencia grande?
2. Cambie `learning_rate` a 0.0001, 0.001 y 0.01 y grafique `plot_loss()`: ¿cuándo diverge el entrenamiento?
3. Con 303 pacientes, ¿qué riesgo hay de sobreajustar con 16 neuronas ocultas? Evalúelo con validación cruzada.

---
## ✅ Conclusiones y siguientes pasos

**Correcciones importantes:** (1) `acc` fallaba con NumPy reciente; (2) `dRelu` modificaba en sitio el arreglo `Z1`; (3) el AUC/ROC se calculaba con etiquetas redondeadas en vez de probabilidades; (4) las fórmulas de sensibilidad y especificidad estaban mal; (5) la historia de las redes neuronales mezclaba fechas (las SVM aparecen en los años 90, no en 1960) y se corrigió [5]. Con 303 pacientes y una red pequeña, interprete las métricas con cautela y valide con validación cruzada [13].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [11].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** McCulloch, W. S., & Pitts, W. (1943). A logical calculus of the ideas immanent in nervous activity. *The Bulletin of Mathematical Biophysics, 5*(4), 115–133. https://doi.org/10.1007/BF02478259

**[4]** Rosenblatt, F. (1958). The perceptron: A probabilistic model for information storage and organization in the brain. *Psychological Review, 65*(6), 386–408. https://doi.org/10.1037/h0042519

**[5]** Minsky, M., & Papert, S. (1969). *Perceptrons: An Introduction to Computational Geometry*. MIT Press.

**[6]** Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature, 323*(6088), 533–536. https://doi.org/10.1038/323533a0

**[7]** Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *Advances in Neural Information Processing Systems, 25*, 1097–1105.

**[8]** Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. https://www.deeplearningbook.org

**[9]** Detrano, R., Janosi, A., Steinbrunn, W., Pfisterer, M., Schmid, J.-J., Sandhu, S., … Froelicher, V. (1989). International application of a new probability algorithm for the diagnosis of coronary artery disease. *The American Journal of Cardiology, 64*(5), 304–310. https://doi.org/10.1016/0002-9149(89)90524-9

**[10]** Dua, D., & Graff, C. (2019). *UCI Machine Learning Repository*. University of California, Irvine. https://archive.ics.uci.edu

**[11]** Murphy, K. P. (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press.

**[12]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[13]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[14]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[15]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html